# Walkthrough: telemetry to anomaly scores

Runs the full pipeline on a small dataset. Follows the lesson order in `docs/`. Cells 1 to 5 need only numpy, pandas and scikit-learn; cells 6 and 7 need PennyLane.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qdc_anomaly.config import DataConfig, FeatureConfig, QuantumConfig
from qdc_anomaly.data.synthetic import METRIC_COLUMNS, generate_telemetry, summarize
from qdc_anomaly.features.windows import make_windows
from qdc_anomaly.features.reduce import QubitReducer
from qdc_anomaly.detectors import build_detector
from qdc_anomaly.evaluation.metrics import compute_metrics, per_fault_recall
from qdc_anomaly.evaluation.benchmark import split_windows

## 1. Generate telemetry (lesson 2)

In [ ]:
data_cfg = DataConfig(n_racks=4, n_steps=720, anomaly_fraction=0.1, seed=3)
telemetry = generate_telemetry(data_cfg)
print(summarize(telemetry))
telemetry.head()

Plot one rack. Shaded regions are injected faults.

In [ ]:
rack = telemetry[telemetry.rack_id == 'rack-00'].reset_index(drop=True)
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, col in zip(axes, ['cpu_util', 'inlet_temp_c', 'fan_rpm', 'mem_util']):
    ax.plot(rack[col], lw=1)
    ax.set_ylabel(col)
    ax.fill_between(rack.index, rack[col].min(), rack[col].max(), where=rack.label == 1, alpha=0.2, color='red')
plt.tight_layout()

## 2. Windows and labels (lesson 3)

In [ ]:
feat_cfg = FeatureConfig(window_size=10, stride=2, n_components=4)
windows = make_windows(telemetry, feat_cfg, METRIC_COLUMNS)
print(windows.X.shape, 'windows x features')
print('anomalous', windows.y.sum(), 'ambiguous', windows.ambiguous.sum(), 'normal', ((windows.y == 0) & ~windows.ambiguous).sum())

## 3. Split and reduce to four angles

In [ ]:
train, test = split_windows(windows, n_train=80, n_test=200, seed=0)
reducer = QubitReducer(n_components=4).fit(train.X)
X_train, X_test = reducer.transform(train.X), reducer.transform(test.X)
print('variance kept', reducer.explained_variance_ratio.sum().round(3))
print('angle range', X_train.min().round(2), X_train.max().round(2))

## 4. Classical baselines (lesson 4)

In [ ]:
q_cfg = QuantumConfig(n_qubits=4, n_layers=2, n_latent=2, epochs=30, batch_size=16, learning_rate=0.05)
scores = {}
for name in ['isolation_forest', 'one_class_svm', 'pca_reconstruction']:
    det = build_detector(name, q_cfg).fit(X_train)
    scores[name] = det.score(X_test)
    m = compute_metrics(test.y, scores[name])
    print(f'{name:20s} ROC AUC {m.roc_auc:.3f}  PR AUC {m.pr_auc:.3f}  F1@10% {m.f1_at_budget:.3f}')

## 5. Quantum kernel SVM (lesson 5)

About a minute on a laptop: 80 x 80 / 2 + 200 x 80 circuits.

In [ ]:
det = build_detector('quantum_kernel_svm', q_cfg).fit(X_train)
scores['quantum_kernel_svm'] = det.score(X_test)
m = compute_metrics(test.y, scores['quantum_kernel_svm'])
print(f'ROC AUC {m.roc_auc:.3f}  PR AUC {m.pr_auc:.3f}  F1@10% {m.f1_at_budget:.3f}')

K = det.kernel_matrix(X_train[:20], X_train[:20])
plt.imshow(K, cmap='viridis'); plt.colorbar(); plt.title('quantum kernel, 20 training windows')

## 6. Quantum autoencoder (lesson 6)

In [ ]:
det = build_detector('quantum_autoencoder', q_cfg, seed=0)
det.verbose = True
det.fit(X_train)
scores['quantum_autoencoder'] = det.score(X_test)
m = compute_metrics(test.y, scores['quantum_autoencoder'])
print(f'ROC AUC {m.roc_auc:.3f}  PR AUC {m.pr_auc:.3f}  F1@10% {m.f1_at_budget:.3f}')
plt.plot(det.history); plt.xlabel('epoch'); plt.ylabel('mean trash probability')

## 7. Compare per fault (lesson 7)

In [ ]:
rows = {name: per_fault_recall(test.y, s, test.fault_type) for name, s in scores.items()}
pd.DataFrame(rows).T.round(2)